# 01 — SentencePiece and BM25F Search

**Question.** Which SentencePiece representation and BM25F parameters provide the strongest lexical retrieval?

Model selection uses only the `tune` queries. The reserved `confirm` split is shown once for the selected variant.


## Setup


In [ ]:
from pathlib import Path
import os, sys, json, time, copy, itertools, importlib.util, subprocess

def locate_project(start=Path.cwd()):
    for parent in [start.resolve(), *start.resolve().parents]:
        if (parent / "pyproject.toml").exists() and (parent / "src/avito_retriever").exists():
            return parent
    candidate = Path("/content/avito-retriever")
    if candidate.exists():
        return candidate
    raise FileNotFoundError("Clone/open avito-retriever or change PROJECT_ROOT in this cell")

PROJECT_ROOT = locate_project()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

def module_available(name):
    try:
        return importlib.util.find_spec(name) is not None
    except ModuleNotFoundError:
        return False

IN_COLAB = module_available("google.colab")
if IN_COLAB and os.environ.get("AVITO_AUTO_INSTALL", "1") != "0":
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "-e",
        f"{PROJECT_ROOT}[lexical,neural,dev]",
    ])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

from avito_retriever.experiments.notebook import resolve_data_dir, result_dir, highlight_best, save_json, save_yaml

DATA_DIR = resolve_data_dir(PROJECT_ROOT)
SEED = 42
np.random.seed(SEED)
print("Project:", PROJECT_ROOT)
print("Data:", DATA_DIR)


In [ ]:
from avito_retriever.config import load_config
from avito_retriever.data.io import load_calibration
from avito_retriever.preprocessing.html import FIELD_COLUMNS
from avito_retriever.preprocessing.normalize import normalize_lexical
from avito_retriever.tokenization.sentencepiece import train_or_load
from avito_retriever.retrieval.bm25f import BM25FIndex
from avito_retriever.evaluation.metrics import evaluate_rankings
from avito_retriever.evaluation.selection import make_tune_confirm_split, metric_on_split

OUT = result_dir(PROJECT_ROOT, "01_sentencepiece_bm25f_search")
parsed = pd.read_parquet(PROJECT_ROOT / "artifacts/cache/parsed_articles.parquet")
calibration = load_calibration(DATA_DIR)
split = make_tune_confirm_split(calibration, n_folds=5, confirm_fold=0, seed=SEED)
split.to_csv(OUT / "selection_split.csv", index=False)
base = load_config(PROJECT_ROOT / "configs/experiments/baseline_bm25f.yaml")


## Search space


In [ ]:
MODEL_TYPES = ["unigram", "bpe"]
VOCAB_SIZES = [4000, 8000, 16000]
K1_VALUES = [0.8, 1.2, 1.5, 2.0]
TITLE_WEIGHTS = [3.0, 5.0, 8.0]

normalization = base["preprocessing"]["normalization"]
lexical = parsed.copy()
for field in FIELD_COLUMNS:
    lexical[field] = lexical[field].fillna("").map(lambda x: normalize_lexical(x, normalization))
queries = calibration.copy()
queries["query_text"] = queries.query_text.map(lambda x: normalize_lexical(x, normalization))
training_texts = sum((lexical[field].tolist() for field in FIELD_COLUMNS), []) + queries.query_text.tolist()
print(f"Trials: {len(MODEL_TYPES)*len(VOCAB_SIZES)*len(K1_VALUES)*len(TITLE_WEIGHTS)}")


## Run resumable grid search


In [ ]:
leaderboard_path = OUT / "bm25f_trials.csv"
done = pd.read_csv(leaderboard_path) if leaderboard_path.exists() else pd.DataFrame()
done_ids = set(done.trial_id) if not done.empty else set()
rows = done.to_dict("records") if not done.empty else []

for model_type, vocab_size in itertools.product(MODEL_TYPES, VOCAB_SIZES):
    sp_cfg = {**base["sentencepiece"], "model_type": model_type, "vocab_size": vocab_size}
    tokenizer = train_or_load(training_texts, sp_cfg, PROJECT_ROOT / "artifacts/indexes/sentencepiece")
    for k1, title_weight in itertools.product(K1_VALUES, TITLE_WEIGHTS):
        trial_id = f"{model_type}-v{vocab_size}-k{k1}-tw{title_weight}"
        if trial_id in done_ids: continue
        fields = copy.deepcopy(base["retrieval"]["bm25f"]["fields"])
        fields["title"]["weight"] = title_weight
        started = time.perf_counter()
        index = BM25FIndex(fields, tokenizer.encode, k1=k1).fit(lexical)
        ranking = index.retrieve(queries, top_k=100, source=trial_id)
        metrics, per_query = evaluate_rankings(ranking, calibration)
        per_query.to_parquet(OUT / f"per_query_{trial_id}.parquet", index=False)
        rows.append({"trial_id": trial_id, "model_type": model_type, "vocab_size": vocab_size,
                     "k1": k1, "title_weight": title_weight,
                     "tune_map@10": metric_on_split(per_query, split, "ap@10", "tune"),
                     "map@10": metrics["map@10"], "recall@50": metrics["recall@50"],
                     "seconds": time.perf_counter()-started, "model_path": str(tokenizer.model_path)})
        pd.DataFrame(rows).to_csv(leaderboard_path, index=False)

trials = pd.DataFrame(rows).sort_values("tune_map@10", ascending=False).reset_index(drop=True)
display(highlight_best(trials.head(15), ["tune_map@10", "map@10", "recall@50"]))


## Select on tune, confirm once


In [ ]:
best = trials.iloc[0].to_dict()
best_per_query = pd.read_parquet(OUT / f"per_query_{best['trial_id']}.parquet")
confirm_map = metric_on_split(best_per_query, split, "ap@10", "confirm")
selection = pd.DataFrame([{"selected_trial": best["trial_id"], "tune_map@10": best["tune_map@10"],
                           "confirm_map@10": confirm_map, "overall_map@10": best["map@10"],
                           "recall@50": best["recall@50"]}])
display(selection)


## Rebuild and save selected lexical ranking


In [ ]:
tokenizer = train_or_load(training_texts, {**base["sentencepiece"], "model_type": best["model_type"],
                           "vocab_size": int(best["vocab_size"])}, PROJECT_ROOT / "artifacts/indexes/sentencepiece")
fields = copy.deepcopy(base["retrieval"]["bm25f"]["fields"])
fields["title"]["weight"] = float(best["title_weight"])
best_index = BM25FIndex(fields, tokenizer.encode, k1=float(best["k1"])).fit(lexical)
best_ranking = best_index.retrieve(queries, top_k=100, source="bm25f_best")
best_ranking.to_parquet(OUT / "best_bm25f_rankings.parquet", index=False)
best_per_query.to_parquet(OUT / "best_bm25f_per_query.parquet", index=False)
best_config = {"sentencepiece": {"model_type": best["model_type"], "vocab_size": int(best["vocab_size"])},
               "bm25f": {"k1": float(best["k1"]), "fields": fields}}
save_yaml(best_config, OUT / "best_bm25f_config.yaml")
save_json({"sentencepiece_model": str(tokenizer.model_path)}, OUT / "best_artifacts.json")


## Visual sensitivity check


In [ ]:
pivot = trials.groupby(["vocab_size", "k1"])["tune_map@10"].max().unstack()
display(pivot.style.format("{:.4f}").highlight_max(axis=None, color="#c6efce"))
pivot.plot(marker="o", figsize=(8,4), title="Best tune MAP@10 by vocabulary and k1")
plt.ylabel("MAP@10"); plt.tight_layout(); plt.show()


## Decision rule

The selected BM25F configuration is frozen in `best_bm25f_config.yaml`. Notebook 02 uses that ranking unchanged while testing dense, kNN and fusion.
